In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")

In [4]:
import torch
import json
import os
import numpy as np
import argparse
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, DOMAIN_PHRASES
from collections import OrderedDict
from vllm import TokensPrompt, SamplingParams, LLM
from pathlib import Path
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

INFO 04-09 22:59:06 __init__.py:190] Automatically detected platform cuda.


In [5]:
cur_dir = Path(".").absolute()

def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)


In [7]:
def extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=True):
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))


In [8]:
def create_hook(phrases, action_phrases, masks_batch_combined, mean_reprs, mean_actions, mean_predicates, combined_len, block_size=4096, scale=1):
    def hook(module, input, output):
        meta = getattr(module, "_meta", {})
        meta["mask_offset"] = meta.get("mask_offset", 0)
        
        if meta["mask_offset"] >= combined_len:
            return output
        
        mask_start = meta["mask_offset"]
        mask_end = mask_start + block_size
        
        meta["mask_offset"] = mask_end
        module._meta = meta
        
        hs, res = output
      
        for ip, phrase in enumerate(phrases):
            if phrase in action_phrases:
                adjustment = mean_actions
            else:
                adjustment = mean_predicates
            
            steering_mask = masks_batch_combined[phrase]
            
            steering_mask = steering_mask[mask_start:mask_end]
            steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
            
            steering_vector = mean_reprs[phrase] - adjustment
            steering_vector = steering_mask[:, None] * steering_vector
            steering_vector = torch.tensor(steering_vector, dtype=hs.dtype, device=hs.device)
        
            hs += steering_vector * scale
        return hs, res
    
    return hook

In [9]:
def add_hook(module, hook_fn):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    module.register_forward_hook(hook_fn)

In [46]:
def remove_hooks(module):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    

In [10]:
block_size = 4096

In [21]:
domain_number = 2

In [22]:
# Load tokenizer
tokenizer = initialize_tokenizer(model_id)

# Load dataset
dataset_name = f"dmitriihook/qwq-32b-planning-mystery-{domain_number}-24k-greedy"

dataset = load_dataset(dataset_name)["train"]

# Load representations
repr_file = f"mystery_representations_greedy/mystery_{domain_number}/mean_reprs_mystery_{domain_number}.json"

In [23]:
with open(repr_file, 'r') as f:
    reprs = json.load(f)

In [24]:
mean_reprs = {k: np.array(v) for k, v in reprs["mean_reprs"].items()}
mean_actions = np.array(reprs["mean_actions"])
mean_predicates = np.array(reprs["mean_predicates"])

In [25]:
domain_key = f"mystery_{domain_number}"
phrases = DOMAIN_PHRASES[domain_key]
action_phrases = list(phrases["actions"].values())
phrases = list(phrases["actions"].values()) + list(phrases["predicates"].values())

In [18]:
llm = LLM(
    model=model_id, 
    tensor_parallel_size=4, 
    enforce_eager=True, 
    max_seq_len_to_capture=20000, 
    max_num_batched_tokens=4096
)

INFO 04-09 23:03:01 config.py:542] This model supports multiple tasks: {'generate', 'reward', 'classify', 'embed', 'score'}. Defaulting to 'generate'.
INFO 04-09 23:03:01 config.py:1401] Defaulting to use mp for distributed inference
WARNING 04-09 23:03:01 arg_utils.py:1135] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 04-09 23:03:01 config.py:1556] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 04-09 23:03:01 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 04-09 23:03:01 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 04-09 23:03:01 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: mode

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


(VllmWorkerProcess pid=603288) INFO 04-09 23:03:21 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=603296) INFO 04-09 23:03:21 model_runner.py:1115] Loading model weights took 15.3937 GB
INFO 04-09 23:03:21 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=603291) INFO 04-09 23:03:21 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=603291) INFO 04-09 23:03:25 worker.py:267] Memory profiling takes 3.13 seconds
(VllmWorkerProcess pid=603291) INFO 04-09 23:03:25 worker.py:267] the current vLLM instance can use total_gpu_memory (79.10GiB) x gpu_memory_utilization (0.90) = 71.19GiB
(VllmWorkerProcess pid=603291) INFO 04-09 23:03:25 worker.py:267] model weights take 15.39GiB; non_torch_memory takes 4.13GiB; PyTorch activation peak memory takes 0.38GiB; the rest of the memory reserved for KV Cache is 51.28GiB.
(VllmWorkerProcess pid=603296) INFO 04-09 23:03:25 worker.py:267] Memory profiling 

In [19]:
sampling_params = SamplingParams(
    max_tokens=19000,
    temperature=0,
    top_k=1,
)

In [65]:
all_tokens = []
phrase_masks = {phrase: [] for phrase in phrases}

In [66]:
initial_lines = 40

In [67]:
row_ids = [1, 1]

for i in row_ids:
    row = dataset[i]
    
    # Process text
    text = "\n\n".join(row["generation"].split("\n\n")[:initial_lines])
    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2][0]
    all_tokens.append(tokens)
    
    # Get phrase positions for this row
    phrase_positions = {
        phrase: extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=False)
        for phrase in phrases
    }
    
    # Create phrase masks for this row
    row_phrase_masks = {
        phrase: np.zeros(tokens.shape[0])
        for phrase in phrases
    }
    
    for phrase in phrases:
        positions = phrase_positions[phrase]
        for start, end in positions:
            row_phrase_masks[phrase][start:end] = 1
    
    # Add masks to batch
    for phrase in phrases:
        phrase_masks[phrase].append(row_phrase_masks[phrase])

In [68]:
all_tokens

[tensor([151644,    872,    198,  ...,   4566,  68088,     13]),
 tensor([151644,    872,    198,  ...,   4566,  68088,     13])]

In [71]:
masks_combined = {
    k: np.concatenate(v, axis=0) for k, v in phrase_masks.items()
}

combined_len = masks_combined[phrases[0]].shape[0] if phrases else 0

In [72]:
combined_len

4240

In [74]:
print(dataset[1]["generation"])

Okay, let's tackle this problem step by step. First, I need to understand the initial conditions and the goal clearly. 

The initial conditions are:
- Aura Block B
- Essence Block D
- Block A harmonizes Block C
- Block C harmonizes Block B
- Block D harmonizes Block A
- Nexus is present.

The goal is to have:
- Block A harmonizes Block C (which is already true, so maybe that's just part of the existing conditions)
- Block B harmonizes Block A
- Block C harmonizes Block D.

Wait, actually, looking again, the goal says "Block A harmonizes Block C, Block B harmonizes Block A and Block C harmonizes Block D." Since Block A harmonizes Block C is already in the initial conditions, maybe that's just part of the required final state, but perhaps it's already satisfied. The main things to achieve are the other two: B harmonizes A and C harmonizes D.

So the problem is to get from the initial state to the goal state by applying the allowed actions (Illuminate, Distill, Silence, Divest) following 

In [75]:
current_hook = create_hook(
    phrases=phrases,
    action_phrases=action_phrases,
    masks_batch_combined=masks_combined,
    mean_reprs=mean_reprs,
    mean_actions=mean_actions,
    mean_predicates=mean_predicates,
    combined_len=combined_len,
    block_size=block_size,
    scale=0,
)

In [76]:
# Apply hook to model
llm.apply_model(
    lambda x: add_hook(x.model.layers[47], current_hook),
)

# llm.apply_model(
#     lambda x: remove_hooks(x.model.layers[47]),
# )

# Create prompts for each row in batch
prompts = [TokensPrompt(prompt_token_ids=tokens.tolist()) for tokens in all_tokens]

# Generate
results2 = llm.generate(prompts, sampling_params=sampling_params)
    

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 2/2 [07:19<00:00, 219.60s/it, est. speed input: 9.65 toks/s, output: 84.50 toks/s]


In [77]:
print(results[0].outputs[0].text)

 The Divest C from B requires:

- C harmonizes B (yes)
- Essence C (yes)
- Nexus (yes)

After Divesting C from B:

- Pulse C becomes true
- Essence B becomes true (since the other is B)
- Removes C harmonizes B
- Removes Essence C (so C loses essence)
- Removes Nexus (so Nexus is gone)

So after this, Essence B is now true, which is good because we need that for Distill D from B.

But now, Nexus is gone. To get Nexus back, we can Silence some object that has Pulse.

In this case, after Divesting, C has Pulse. So we can Silence C.

Silence C requires Pulse C (yes). After Silence:

- Essence, Aura, Nexus become true again (for C?), or for the object being silenced?

Wait, the Silence action's effects:

Once Silence is performed on object:

- True: Essence object, Aura object, Nexus (wait, the problem says "Once Silence action is performed the following will be true: Essence object, Aura object, Nexus." So it's for the object being silenced?

Wait, the problem says:

"For Silence action, 

In [51]:
print(results2[0].outputs[0].text)

 The Divest C from B requires:

- C harmonizes B (yes)
- Essence C (yes)
- Nexus (yes)

After Divesting C from B:

- Pulse C becomes true
- Essence B becomes true (since the other is B)
- Removes C harmonizes B
- Removes Essence C (so C loses essence)
- Removes Nexus (so Nexus is gone)

So after this, Essence B is now true, which is good because we need that for Distill D from B.

But now, Nexus is gone. To get Nexus back, we can perform Silence on an object with Pulse. Since after Divesting, C has Pulse (from the Divest effect), so we can Silence C.

Silence C requires Pulse C (yes). After Silence:

- Essence, Aura, Nexus become true again (so Nexus is back)
- Pulse C becomes false.

So after Silence C, we have:

- Essence B is still true (from Divest)
- Nexus is back
- C's essence is gone (from Divest)
- C's Pulse is now false (after Silence)
- The harmonize between C and B is gone (so we lost that, but the goal requires B harmonizes C again? Wait, the goal says "Block B harmonizes B